In [ ]:
from pathlib import Path
import glob

from sparkrawl import *
from ospath import *
from fileio import *
from testcases import *

In [ ]:
import tempfile
tmpdir = tempfile.TemporaryDirectory()

tmp_path = Path(tmpdir.name)

In [ ]:
from fakes3 import *

fake_s3 = FakeS3(tmp_path / "fake-s3")

def list_all(uri):
    return [
        *(uri_ for uri_ in fake_s3.list_objects(uri)),
        *(
            uri__
            for uri_ in fake_s3.list_prefixes(uri)
            for uri__ in list_all(uri_)            
        ),
    ]

In [ ]:
# Populate a directory with data, organized according to path-embedded metadata.
import random

from fileio import write_jsonlines

COLORS = ["green", "red", "blue"]
YEARS = [2024, 2025]
SIZES = ["large", "small"]

def populate_directory(path: Path):

    for c in COLORS:
        for y in YEARS:
            for sz in SIZES:
                for k in range(4):
                    child = path / c / str(y) / f"size={sz}"
                    child.mkdir(parents=True, exist_ok=True)

                    file = child / f"{k}.jsonl"
                    data = [{"x": random.random()} for _ in range(10)]

                    write_jsonlines(data, file)

data_root_uri = "s3://fake-bucket/fake-prefix"
populate_directory(fake_s3._resolve(data_root_uri))

list_all(data_root_uri)

In [ ]:
fake_s3.list_prefixes(data_root_uri)

In [ ]:
# We might want to use the color metadata encoded in the path.
def list_prefixes_with_color_metadata(uri):
    """This method knows how to interpet the directory name at this level as metadata."""
    return [
        (prefix, dict(color=uri_name(prefix)))
        for prefix in fake_s3.list_prefixes(uri)
    ]
    
def uri_name(uri: str):
    return Uri.from_uri(uri).key_path.name

list_prefixes_with_color_metadata(data_root_uri)

In [ ]:
# If we "crawl", we usually want to collect all the metadata along a given path, perhaps including some additional global attributes.

def list_prefixes_with_year_metadata(uri):
    return [
        (prefix, dict(year=int(uri_name(prefix))))
        for prefix in fake_s3.list_prefixes(uri)
    ]

def list_prefixes_with_color_and_year_metadata(tagged):
    prefix, base_attrs = tagged
    return [
        (year_prefix, {**base_attrs, **color_attrs, **year_attrs})
        for color_prefix, color_attrs in list_prefixes_with_color_metadata(prefix)
        for year_prefix, year_attrs in list_prefixes_with_year_metadata(color_prefix)
    ]

tagged = data_root_uri, {"global_attr": 42}
list_prefixes_with_color_and_year_metadata(tagged)

In [ ]:
# This approach can get out of hand quickly with many levels of nesting.
# Creating these crawlers can be time consuming and error-prone.

# One improvement..
# Work with what we will call "exploders".
# An exploder is also a tagged child iterator, but takes a _tagged_ parent element as input, rather than the parent element by itself.
# This creates input-output symmetry that allows parent metadata to be passed along crawler chains.

def as_exploder(key_iter_fn):
    
    def exploder(tagged):
        key, attrs = tagged
        return key_iter_fn(key)

    return exploder

color_exploder = as_exploder(list_prefixes_with_color_metadata)
year_exploder = as_exploder(list_prefixes_with_year_metadata)

# The main contributions of the library is the `explode_with` function, which decorates exploders to handle attribute collection.

[
    tagged__
    for tagged_ in explode_with(color_exploder)(tagged)
    for tagged__ in explode_with(year_exploder)(tagged_)
]

In [ ]:
# Here's what the same logic would look like in Spark.

import pysparkling
sc = pysparkling.Context()

rdd = sc.parallelize([tagged])

rdd.flatMap(explode_with(color_exploder)).flatMap(explode_with(year_exploder)).collect()

In [ ]:
# A convenience function is provided to simulate flatMap chains locally.
# or possibly to group stages to reduce the number of flatMaps in a chain.
crawler = fan_out(
    explode_with(color_exploder),
    explode_with(year_exploder),
)
list(crawler(tagged))

In [ ]:
# The library also supplies functional utilities that help combine functions into exploders without writing so many custom functions.
# They are not mandatory, but they are available.

# Build branch-level exploders from primitives, without ad hoc function defs.
color_exploder = pipeline(key_only, fake_s3.list_prefixes, for_each(with_attribs(color=uri_name)))
year_exploder = pipeline(key_only, fake_s3.list_prefixes, for_each(with_attribs(year=pipeline(uri_name, int))))

rdd.flatMap(explode_with(color_exploder)).flatMap(explode_with(year_exploder)).collect()

In [ ]:
# We just introduced `pipeline`. What is it? Pretty easy.
# e.g., also seen as https://toolz.readthedocs.io/en/latest/api.html#toolz.functoolz.compose_left

assert pipeline(range, sum, lambda x: x ** 2)(4) == sum(range(4)) ** 2

In [ ]:
# key_only is easy. Another way to define `as_exploder` is in terms of `key_only`.
as_exploder_ = lambda key_iter_fn: pipeline(key_only, key_iter_fn)

color_exploder_ = as_exploder_(list_prefixes_with_color_metadata)

assert list(color_exploder_(tagged)) == list(color_exploder(tagged))

In [ ]:
# `with_attrs` also pretty easy
with_strlen = with_attribs(dict(strlen=lambda string: len(string)))

print(with_strlen("hello"))

# and `for_each`
each_with_strlen = for_each(with_strlen)

print(list(each_with_strlen(["hello", "again!"])))

# a few other useful utilities in the API docs

In [ ]:
# So here's a full data RDD in Spark.

def parquet_attribs(uri):
    """
    "s3://..../size=small" -> {"size": "small"}
    """
    attrib, value = uri_name(uri).split("=", maxsplit=1)
    return {attrib: value}

parquet_exploder = pipeline(key_only, fake_s3.list_prefixes, for_each(compute_value(parquet_attribs)))
partition_exploder = pipeline(key_only, fake_s3.list_objects, for_each(with_attribs()))

def read_s3_json(uri):
    with tempfile.NamedTemporaryFile() as f:
        fake_s3.get(uri, f.name)
        yield from read_jsonlines(Path(f.name))

record_exploder = pipeline(key_only, read_s3_json, for_each(key_by_none))

(rdd
    .flatMap(explode_with(color_exploder))
    .flatMap(explode_with(year_exploder))
    .flatMap(explode_with(parquet_exploder))
    .flatMap(explode_with(partition_exploder))
    .flatMap(explode_with(record_exploder))
    .map(drop_key)
).take(5)

In [ ]:
# A nice thing about this approach is that stages can easily be split up.
# For example, often we crawl for files in one phase, then process the data in another.
# Sometimes we even collect URIs to redistribute them more evenly to workers.

# Crawl to collect all data files.
files_with_metadata = (rdd
    .flatMap(fan_out(
        explode_with(color_exploder),
        explode_with(year_exploder),
        explode_with(parquet_exploder),
        explode_with(partition_exploder)
    ))
).collect()

def take(coll, n):
    return [
        i for i, _ in zip(coll, range(n))
    ]

print(take(files_with_metadata, 5))

# Redistribute data files and extract contents.
tagged_files_rdd = sc.parallelize(files_with_metadata)
record_rdd = tagged_files_rdd.flatMap(explode_with(record_exploder)).map(drop_key)

# Show a random sample.
count = record_rdd.count()
p = 10 / count
record_rdd.sample(withReplacement=False, fraction=p).collect()[:5]